In [1]:
from dotenv import load_dotenv
import os
import voyageai

from qdrant_client import QdrantClient

/Users/pranjal/Desktop/backup projects/AI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
load_dotenv("../../.env")

True

### Embedding Function

In [6]:
load_dotenv()
VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

def get_embedding(text, model = 'voyage-3'):
        result = vo.embed(
                [text],
                model=model,
                input_type="document"
        )
        return result.embeddings[0]

### Retrieval Function

In [7]:
qdrant_client = QdrantClient(url = "http://localhost:6333")

In [19]:
def retrieve_data(query, qdrant_client, k=5):
        query_embedding = get_embedding(query)
        results = qdrant_client.query_points(
                collection_name="Amazon-items-collection-00",
                query=query_embedding,
                limit=k,
        )

        retrieved_context_ids = []
        retrieved_context = []
        similarity_scores = []
        retrieved_context_ratings = []

        for item in results.points:
                retrieved_context_ids.append(item.payload['parent_asin'])
                retrieved_context.append(item.payload['description'])
                retrieved_context_ratings.append(item.payload['average_rating'])
                similarity_scores.append(item.score)
        
        return {
                "retrieved_context_ids": retrieved_context_ids,
                "retrieved_context": retrieved_context,
                "retrieved_context_ratings": retrieved_context_ratings,
                "similarity_scores": similarity_scores,
        }

In [20]:
retrieve_context = retrieve_data("What kind of earphones can I get", qdrant_client, 10)

In [21]:
retrieve_context

{'retrieved_context_ids': ['B09F36P17Y',
  'B0B4WNFTVZ',
  'B0BS1GQJ5S',
  'B0C32SMPNP',
  'B0BCKCJQPN',
  'B0BJ9PRHZ3',
  'B09ZWNVYV2',
  'B0BG29CCQ3',
  'B09TKF5S6W',
  'B0C9X4HB3H'],
 'retrieved_context': ['Wireless Earbud, Bluetooth 5.1 Headphones with Microphone Deep Bass Bluetooth Earphones in-Ear, CVC8.0 Noise Cancelling Earbud for Sport Running Gym IPX7 Waterproof, 30H Playtime, Touch Control White ',
  '2022 Updated Digital TV Antenna to 500 Miles Range ',
  'Togkun 2 Packs Earbuds Headphones Wired—Corded Ear Buds with Built-in Mic and Headphone Comfortable in-Ear Audifonos Earphones 【Save Time & Energy】 Ultra-portable Memory Function allows you just to turn on the Bluetooth before plugging in your earbuds for the first time. Then click "Connect with iCloud". They will connect automatically within 2-5 seconds next time if you plug in the earbuds feufain.',
  "Monster Open Ear Headphones Air Conduction Headphones Wireless Sport Headphones Bluetooth 5.3, Running Earphones with E

### Format retrieved context data

In [27]:
def process_context(context):
        formatted_context = ""
        for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
                formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        return formatted_context

In [28]:
preprocessed_context = process_context(retrieve_context)

In [29]:
print(preprocessed_context)

- ID: B09F36P17Y, rating: 4.1, description: Wireless Earbud, Bluetooth 5.1 Headphones with Microphone Deep Bass Bluetooth Earphones in-Ear, CVC8.0 Noise Cancelling Earbud for Sport Running Gym IPX7 Waterproof, 30H Playtime, Touch Control White 
- ID: B0B4WNFTVZ, rating: 3.4, description: 2022 Updated Digital TV Antenna to 500 Miles Range 
- ID: B0BS1GQJ5S, rating: 3.5, description: Togkun 2 Packs Earbuds Headphones Wired—Corded Ear Buds with Built-in Mic and Headphone Comfortable in-Ear Audifonos Earphones 【Save Time & Energy】 Ultra-portable Memory Function allows you just to turn on the Bluetooth before plugging in your earbuds for the first time. Then click "Connect with iCloud". They will connect automatically within 2-5 seconds next time if you plug in the earbuds feufain.
- ID: B0C32SMPNP, rating: 3.7, description: Monster Open Ear Headphones Air Conduction Headphones Wireless Sport Headphones Bluetooth 5.3, Running Earphones with Enhanced Bass, IPX5 Sweatproof for Workout, 7H Pla

### Create Prompt function

In [37]:
def build_pompt(preprocessed_context, question):
        prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products

Context:
{preprocessed_context}

Question:
{question}
        """
        return prompt

In [38]:
prompt = build_pompt(preprocessed_context, "What kind of earphones can I get?")
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products

Context:
- ID: B09F36P17Y, rating: 4.1, description: Wireless Earbud, Bluetooth 5.1 Headphones with Microphone Deep Bass Bluetooth Earphones in-Ear, CVC8.0 Noise Cancelling Earbud for Sport Running Gym IPX7 Waterproof, 30H Playtime, Touch Control White 
- ID: B0B4WNFTVZ, rating: 3.4, description: 2022 Updated Digital TV Antenna to 500 Miles Range 
- ID: B0BS1GQJ5S, rating: 3.5, description: Togkun 2 Packs Earbuds Headphones Wired—Corded Ear Buds with Built-in Mic and Headphone Comfortable in-Ear Audifonos Earphones 【Save Time & Energy】 Ultra-portable Memory Function allows you just to turn on the Bluetooth before plugging in your earbuds for the first time. Then click "Connect with iCloud". They will con

In [49]:
import anthropic
client = anthropic.Anthropic()

def generate_answer(prompt):
        message = client.messages.create(
                max_tokens=2000,
                messages=[
                        {
                        "role": "user",
                        "content": prompt,
                        }
                ],
                model="claude-opus-4-7",
        )
        return message.content[0].text

In [46]:
print(generate_answer(prompt))

Based on the available products, here are the earphones/earbuds you can choose from:

---

### Wireless Earbuds

1. **Wireless Earbud, Bluetooth 5.1** (ID: B09F36P17Y) ⭐ 4.1
   - Deep bass, CVC8.0 noise cancelling, IPX7 waterproof, 30H playtime, touch control, built-in microphone — great for sports/running/gym. Available in **White**.

2. **Soundcore by Anker Space A40** (ID: B0BJ9PRHZ3) ⭐ 4.3
   - Active Noise Cancelling (reduces noise up to 98%), **50H playtime**, Hi-Res sound with LDAC, wireless charging, comfortable lightweight design, app customization.

3. **Bluetooth True Wireless Earbuds** (ID: B09TKF5S6W) ⭐ 3.8
   - LED power display, 30H playtime, wireless charging case, IPX4 waterproof, Bluetooth 5.0, lightweight ergonomic design.

4. **Monster Open Ear Headphones** (ID: B0C32SMPNP) ⭐ 3.7
   - Open-ear design using bone & air conduction technology, Bluetooth 5.3, IPX5 sweatproof, 7H playtime, 60ms low latency gaming mode — ideal for workouts. Available in **Silver**.

---

#

### Combined RAG Pipeline

In [52]:
def rag_pipeline(question, top_k=10):
        qdrant_client = QdrantClient(url='http://localhost:6333')
        retrieved_context = retrieve_data(question, qdrant_client, top_k)
        preprocessed_context = process_context(retrieved_context)
        prompt = build_pompt(preprocessed_context, question)
        answer = generate_answer(prompt)

        return answer

In [53]:
print(rag_pipeline("What kind of earphone can I get with rating above 4.5"))

Based on the available products, there are no earphones with a rating above 4.5.

The only product with a rating above 4.5 is the **JBL GO 3 Waterproof Ultra Portable Bluetooth Speaker Bundle** (ID: B09ZWNVYV2) with a rating of 4.7, but this is a Bluetooth speaker, not an earphone.

The highest-rated earphones available are:
- **Soundcore by Anker Space A40** (ID: B0BJ9PRHZ3) — rating 4.3, active noise cancelling wireless earbuds with up to 50H playtime.
- **Wireless Earbud, Bluetooth 5.1 Headphones** (ID: B09F36P17Y) — rating 4.1, in-ear earbuds with deep bass, CVC8.0 noise cancelling, IPX7 waterproof, and 30H playtime.

Would you like more details on either of these?
